In [52]:
import unittest
from collections.abc import Iterable, Sized, Sequence, Mapping
from cerberus import errors, validator

_str_type = str

sample_schema = {
    'a_string': {'type': 'string', 'minlength': 2, 'maxlength': 10},
    'a_binary': {'type': 'binary', 'minlength': 2, 'maxlength': 10},
    'a_nullable_integer': {'type': 'integer', 'nullable': True},
    'an_integer': {'type': 'integer', 'min': 1, 'max': 100},
    'a_restricted_integer': {'type': 'integer', 'allowed': [-1, 0, 1]},
    'a_boolean': {'type': 'boolean', 'meta': 'can haz two distinct states'},
    'a_datetime': {'type': 'datetime', 'meta': {'format': '%a, %d. %b %Y'}},
    'a_float': {'type': 'float', 'min': 1, 'max': 100},
    'a_number': {'type': 'number', 'min': 1, 'max': 100},
    'a_set': {'type': 'set'},
    'one_or_more_strings': {'type': ['string', 'list'], 'schema': {'type': 'string'}},
    'a_regex_email': {
        'type': 'string',
        'regex': r'^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+$',
    },
    'a_readonly_string': {'type': 'string', 'readonly': True},
    'a_restricted_string': {'type': 'string', 'allowed': ['agent', 'client', 'vendor']},
    'an_array': {'type': 'list', 'allowed': ['agent', 'client', 'vendor']},
    'an_array_from_set': {
        'type': 'list',
        'allowed': set(['agent', 'client', 'vendor']),
    },
    'a_list_of_dicts': {
        'type': 'list',
        'schema': {
            'type': 'dict',
            'schema': {
                'sku': {'type': 'string'},
                'price': {'type': 'integer', 'required': True},
            },
        },
    },
    'a_list_of_values': {
        'type': 'list',
        'items': [{'type': 'string'}, {'type': 'integer'}],
    },
    'a_list_of_integers': {'type': 'list', 'schema': {'type': 'integer'}},
    'a_dict': {
        'type': 'dict',
        'schema': {
            'address': {'type': 'string'},
            'city': {'type': 'string', 'required': True},
        },
    },
    'a_dict_with_valuesrules': {'type': 'dict', 'valuesrules': {'type': 'integer'}},
    'a_list_length': {
        'type': 'list',
        'schema': {'type': 'integer'},
        'minlength': 2,
        'maxlength': 5,
    },
    'a_nullable_field_without_type': {'nullable': True},
    'a_not_nullable_field_without_type': {},
}

MUMCUT:

1. Test for LIF: Given a minimal DNF representation of a predicate f , for each implicant i, choose unique true points (UTPs) such that clauses not in i take on values T and F.


2. Test for LOF: Given a minimal DNF representation of a predicate f , for each literal c in each implicant i, TR contains a unique true point for i and a near false point for c in i such that the two points differ only in the truth value of c.


3. Test for infeasible cases: Given a minimal DNF representation of a predicate f , for each literal c in each implicant i, choose near false points (NFPs) such that clauses not in i take on values T and F.



**_validate_allowed**

*Requirements:*

1. If the given value is not an iterable (excluding str) checks if it's an instance that is within allowed_values, if not raises an UNALLOWED_VALUES error,
2. If the given value is an iterable but it's also a str checks if it's an instance that is within allowed_values, if not raises an UNALLOWED_VALUES error,
3. If the given value is an iterable (excluding str) checks if the elements inside are instances that are within allowed_values, if at least one isn't raises an UNALLOWED_VALUES error,
4. If none of the conditions are met doesnt raise an error or give an output.

*Logic Equations:*
- $ A = $ "value" is an iterable
- $ B = $ "value" is a str
- $ C = $ "value" is an element of "allowed_values"
- $ D = $ "value" is a subset of "allowed values"
- $ F = $ "_validate_allowed" method does NOT raise an UNALLOWED_VALUES error with the given parameters

$ F = A \bar{B} D +  C(\bar{A} + B)$

Or in a more minimal way:

$ F = A \bar{B} D +  C\bar{A} + CB$

*Test Cases:*

- MUTP:
    - Implicant $ = A \bar{B} D \longrightarrow A = 1, B = 0, D = 1 $
        1. $ABCD = 1001$ 
        2. $ABCD = 1011$ 
    - Implicant $ =  C\bar{A} \longrightarrow A = 0, C = 1 $
        1. $ABCD = 0010$ 
        2. $ABCD = 0011$ 
        3. $ABCD = 0110$ 
        4. $ABCD = 0111$ 
    - Implicant $ =  CB \longrightarrow B = 1, C = 1 $
        1. $ABCD = 0110$ 
        2. $ABCD = 1110$ 
        3. $ABCD = 0111$ 
        4. $ABCD = 1111$ 
- CUTPNFP:
    - Clause $ = A$ and Implicant $ = A \bar{B} D,$ True Points: $ABD = 101, C = 0/1,$ Near false points:
        1. $ABCD = 0001$ 
        2. $ABCD = 0011$ 
    - Clause $ = A$ and Implicant $ = C\bar{A},$ True Points: $AC = 01, BD = 00/01/11/10,$ Near false points:
        1. $ABCD = 1010$ 
        2. $ABCD = 1110$ 
        3. $ABCD = 1011$ 
        4. $ABCD = 1111$
    - Clause $ = B$ and Implicant $ = A \bar{B} D,$ True Points: $ABD = 101, C = 0/1,$ Near false points:
        1. $ABCD = 1101$ 
        2. $ABCD = 1111$ 
    - Clause $ = B$ and Implicant $ = CB,$ True Points: $BC = 11, AD = 00/01/11/10,$ Near false points:
        1. $ABCD = 0010$ 
        2. $ABCD = 1010$ 
        3. $ABCD = 0011$ 
        4. $ABCD = 1011$
    - Clause $ = C$ and Implicant $ = C\bar{A},$ True Points: $AC = 01, BD = 00/01/11/10,$ Near false points:
        1. $ABCD = 0000$ 
        2. $ABCD = 0100$ 
        3. $ABCD = 0001$ 
        4. $ABCD = 0101$
    - Clause $ = C$ and Implicant $ = CB,$ True Points: $BC = 11, AD = 00/01/11/10,$ Near false points:
        1. $ABCD = 0100$ 
        2. $ABCD = 1100$ 
        3. $ABCD = 0101$ 
        4. $ABCD = 1101$
    - Clause $ = D$ and Implicant $ = A \bar{B} D,$ True Points: $ABD = 101, C = 0/1,$ Near false points:
        1. $ABCD = 1000$ 
        2. $ABCD = 1010$ 
- MNFP:
    - Literal $ = A$ and Implicant $ = A \bar{B} D,$ True Points: $ABD = 101, C = 0/1,$ Near false points:
        1. $ABCD = 0001$ 
        2. $ABCD = 0011$ 
    - Literal $ = A$ and Implicant $ = C\bar{A},$ True Points: $AC = 01, BD = 01/10,$ Near false points:
        1. $ABCD = 1110$ 
        2. $ABCD = 1011$
    - Literal $ = B$ and Implicant $ = A \bar{B} D,$ True Points: $ABD = 101, C = 0/1,$ Near false points:
        1. $ABCD = 1101$ 
        2. $ABCD = 1111$ 
    - Literal $ = B$ and Implicant $ = CB,$ True Points: $BC = 11, AD = 01/10,$ Near false points:
        1. $ABCD = 1010$ 
        2. $ABCD = 0011$
    - Literal $ = C$ and Implicant $ = C\bar{A},$ True Points: $AC = 01, BD = 01/10,$ Near false points:
        1. $ABCD = 0100$ 
        2. $ABCD = 0001$ 
    - Literal $ = C$ and Implicant $ = CB,$ True Points: $BC = 11, AD = 01/10,$ Near false points:
        1. $ABCD = 1100$ 
        2. $ABCD = 0101$
    - Literal $ = D$ and Implicant $ = A \bar{B} D,$ True Points: $ABD = 101, C = 0/1,$ Near false points:
        1. $ABCD = 1000$ 
        2. $ABCD = 1010$ 

This should be all the required test cases for MUMCUT, but it should be noted that there is severe overlap between these.

The unique cases within these are found in the code cell below: 

In [53]:
newlis = list()
alllis = "1001,1011,0010,0011,0110,0111,0110,1110,0111,1111,0001,0011,1010,1110,1011,1111,1101,1111,0010,1010,0011,1011,0000,0100,0001,0101,0100,1100,0101,1101,1000,1010"
alllis = alllis.split(",")
for ele in alllis:
    if ele not in newlis:
        newlis.append(ele)
print(newlis)
print(len(newlis))

['1001', '1011', '0010', '0011', '0110', '0111', '1110', '1111', '0001', '1010', '1101', '0000', '0100', '0101', '1100', '1000']
16


I refrained from putting in all the cases i originally found since the newlis length already reached 16, which is the max possible unique values able to be acquired from 4 binary digits.

To ensure MUMCUT requirements all these 16 cases will need to be tested. A possible test program is given below:

In [62]:
# A = "value" is an iterable
# B = "value" is a str
# C = "value" is an element of "allowed_values"
# D = "value" is a subset of "allowed values"
# F = "_validate_allowed" method does NOT raise an UNALLOWED_VALUES error with the given parameters
allowed_values = None
field = 'a_string'
value = None
sample_min_schema = {'a_string': {'type': 'string', 'minlength': 2, 'maxlength': 10}}
val = validator.Validator(sample_schema)
tcase = unittest.TestCase()
for ele in newlis:
    if ele[0] == '0':
        value = 1
    else: value = sample_min_schema
    if ele[1] == '1' and ele[0] == '0':
        value = {}
    elif ele[1] == '1': value = "123"
    if ele[2] == '0':
        allowed_values = {'a_string': {'type': 'string', 'minlength': 2, 'maxlength': 10}}
    else: allowed_values = {'value': value}
    if ele[3] == '1':
        if ele[0] == '1':
            for ele2 in value:
                allowed_values[ele2] = value[ele2]
        else:
            allowed_values.append(value)
    tcase.assertNoLogs("",val._validate_allowed(allowed_values,field,value))


KeyError: 'allowed'

**_validate_max**

*Requirements:*

1. If the given value is greater than the given max_value raises a MAX_VALUE error,
2. If the given value and the max_value are NOT comparable types does NOT raise a TypeError,
3. If neither condition is met doesnt raise an error or give an output.

*Logic Equations:*
- $ A = $ "value" is greater than "max_value"
- $ B = $ "value" and "max_value" are not comparable types
- $ F = $ "_validate_max" method does NOT raise an MAX_VALUE error with the given parameters

$ F = \bar{A}$

The function uses a try/except method to supress TypeErrors so B is not relevant to the final equation. Weird choice by the developer but it works.

*Test Cases:*

- MUTP:
    - Implicant $ = \bar{A} \longrightarrow A = 0 $
        1. $AB = 01$ 
        2. $AB = 00$ 
- CUTPNFP:
    - Clause $ = A$ and Implicant $ = \bar{A},$ True Points: $A = 0, B = 0/1,$ Near false points:
        1. $AB = 11$ 
        2. $AB = 10$
- MNFP:
    - Literal $ = A$ and Implicant $ = \bar{A},$ True Points: $A = 0, B = 0/1,$ Near false points:
        1. $AB = 11$ 
        2. $AB = 10$ 

This function doesn't require many test cases as the equation is expected to depend on only one of the requirements.

Unique cases: 01, 00, 11, 10

In [ ]:
# def _validate_max(self, max_value, field, value):
#         """{'nullable': False }"""
#         try:
#             if value > max_value:
#                 self._error(field, errors.MAX_VALUE)
#         except TypeError:
#             pass

**_validate_contains**

Requirements:
1. If value is not an iterable returns None,
2. If expected_values is an iterable (excluding str) converts expected_values into an unordered list consisting of it's elements, otherwise converts into an unordered list with it's inital value as the only element. Then compares the resulting unordered list to an unordered list of elements within value to determine if expected_values is a subset of value. If expected_values is NOT a subset of value raises a MISSING_MEMBERS error,
3. If neither condition is met doesn't raise an error or give an output.

*Logic Equations*
- $ A = $ "value" is an iterable
- $ B = $ "expected_value" is a non string iterable 
- $ C = $ set created from "expected_values" is a subset of "value"
- $ F = $ "_validate_contains" method does NOT raise an MISSING_MEMBERS error with the given parameters

$ F = AC$

The function uses B only when creating a set from "expected_values", thus it doesnt factor into final equation.

*Test Cases:*

- MUTP:
    - Implicant $ = AC \longrightarrow A = 1, C = 1 $
        1. $ABC = 101$ 
        2. $ABC = 111$ 
- CUTPNFP:
    - Clause $ = A$ and Implicant $ = AC,$ True Points: $AC = 11, B = 0/1,$ Near false points:
        1. $ABC = 001$ 
        2. $ABC = 011$
    - Clause $ = C$ and Implicant $ = AC,$ True Points: $AC = 11, B = 0/1,$ Near false points:
        1. $ABC = 100$ 
        2. $ABC = 110$
- MNFP:
    - Literal $ = A$ and Implicant $ = AC,$ True Points: $AC = 11, B = 0/1,$ Near false points:
        1. $ABC = 001$ 
        2. $ABC = 011$ 
    - Literal $ = C$ and Implicant $ = AC,$ True Points: $AC = 11, B = 0/1,$ Near false points:
        1. $ABC = 100$ 
        2. $ABC = 110$ 



In [ ]:
# def _validate_contains(self, expected_values, field, value):
#         """{'empty': False }"""
#         if not isinstance(value, Iterable):
#             return

#         if not isinstance(expected_values, Iterable) or isinstance(
#             expected_values, _str_type
#         ):
#             expected_values = set((expected_values,))
#         else:
#             expected_values = set(expected_values)

#         missing_values = expected_values - set(value)
#         if missing_values:
#             self._error(field, errors.MISSING_MEMBERS, missing_values)


**_validate_empty**

Requirements:
1. If given "value" parameter provides a len() method and the len() method returns 0 removes the validation checks for the "allowed, forbidden, items, minlength, maxlength, regex, check_with" rules from the queue, then if the given "empty" parameter is False raises an EMPTY_NOT_ALLOWED error.
2. If both conditions aren't met at the same time doesn't raise an error or provide an output.

*Logic Equations*
- $ A = $ "value" provides a len() method & has a length of 0
- $ B = $ "empty" is True
- $ F = $ "_validate_empty" method does NOT raise an EMPTY_NOT_ALLOWED error with the given parameters

$ F = \bar{A} + B$

*Test Cases:*

- MUTP:
    - Implicant $ = \bar{A} + B\longrightarrow A = 0, B = 1 $
        1. $AB = 01$ 
- CUTPNFP:
    - Clause $ = A$ and Implicant $ = \bar{A} + B,$ True Points: $AB = 01,$ Near false points:
        1. $AB = 11$
    - Clause $ = B$ and Implicant $ = \bar{A} + B,$ True Points: $AB = 01,$ Near false points:
        1. $AB = 00$
- MNFP:
    - Literal $ = A$ and Implicant $ = \bar{A} + B,$ True Points: $AB = 01,$ Near false points:
        1. $AB = 11$
    - Literal $ = B$ and Implicant $ = \bar{A} + B,$ True Points: $AB = 01,$ Near false points:
        1. $AB = 00$

Not many test cases since this equation does depend on all requirements.

In [ ]:
# def _validate_empty(self, empty, field, value):
#         """{'type': 'boolean'}"""
#         if isinstance(value, Sized) and len(value) == 0:
#             self._drop_remaining_rules(
#                 'allowed',
#                 'forbidden',
#                 'items',
#                 'minlength',
#                 'maxlength',
#                 'regex',
#                 'check_with',
#             )
#             if not empty:
#                 self._error(field, errors.EMPTY_NOT_ALLOWED)

# # Sized is an abstract class method used to check if an abc provides the len() method

# # def _drop_remaining_rules(self, *rules):
# #         """
# #         Drops rules from the queue of the rules that still need to be evaluated for the
# #         currently processed field. If no arguments are given, the whole queue is
# #         emptied.
# #         """
# #         if rules:
# #             for rule in rules:
# #                 try:
# #                     self._remaining_rules.remove(rule)
# #                 except ValueError:
# #                     pass
# #         else:
# #             self._remaining_rules = []


**_validate_schema**

Requirements:
1. If the given "schema" is None then returns None,
2. If the given "value" parameter is an ordered collection of elements (collections.Sequence) excluding a str calls the __validate_schema_sequence method with it's own input parameters,
3. Otherwise if the given "value" parameter is a collection of elements that provides key lookups (collections.Mapping) calls the __validate_schema_mapping method with it's own input parameters.
4. Does not raise an error or provide an output value.

*Logic Equations*
- $ A = $ "schema" is None
- $ B = $ "value" is a Sequence other than string
- $ C = $ "value" is provides key lookups
- $ F = $ "_validate_empty" method calls either "__validate_schema_sequence" or "__validate_schema_mapping"

$ F = \bar{A}(B + C)$

In [ ]:
# def _validate_schema(self, schema, field, value):
#         """
#         {'type': ['dict', 'string'],
#          'anyof': [{'check_with': 'schema'},
#                    {'check_with': 'bulk_schema'}]}
#         """
#         if schema is None:
#             return

#         if isinstance(value, Sequence) and not isinstance(value, _str_type):
#             self.__validate_schema_sequence(field, schema, value)
#         elif isinstance(value, Mapping):
#             self.__validate_schema_mapping(field, schema, value)
